# Notebook 01 — Asclepius Data Exploration

This notebook is the data-understanding stage of a medical LLM fine-tuning project. We will download the **Asclepius Synthetic Clinical Notes** dataset, inspect its real schema and examples, measure quality and length distributions, select a controlled 5,000-example subset, split it without case leakage, and save a model-independent supervised-learning dataset.

**No model training happens here.** We deliberately keep model choice, chat templates, quantization, LoRA/QLoRA, and training configuration for the next notebook.

The workflow is:

```text
raw dataset
    ↓
dataset inspection
    ↓
quality checks
    ↓
task distribution and case grouping
    ↓
controlled 5,000-example selection
    ↓
case-aware train/validation/test split
    ↓
text-length analysis
    ↓
canonical supervised-learning representation
    ↓
optional tokenizer analysis
    ↓
validation and saved dataset
```

The source is synthetic clinical text. This notebook is not intended for real patient data.

## Why inspect the data before training?

A fine-tuning run can fail for reasons that have nothing to do with the model: unexpected columns, empty targets, duplicated clinical cases across splits, a skewed task mix, or examples that are too long for the eventual context window. Measuring these properties first makes the later experiment reproducible and makes it possible to distinguish a data problem from a modeling problem.

## 1. Install the exploration dependencies

These are the only packages needed for this notebook:

- `datasets` loads and saves Hugging Face datasets without requiring a full pandas conversion.
- `pandas` provides compact tables for summaries.
- `matplotlib` and `seaborn` provide simple plots.
- `transformers` is included only so a tokenizer can be inspected later if the next model is already known; this notebook does **not** download a model.

PyTorch is not installed here. If the Kaggle environment already provides it, we will report its version and CUDA status below.

In [ ]:
%pip install -q datasets pandas matplotlib seaborn transformers

## 2. Imports, environment information, and reproducibility

`SEED` is used consistently for sampling and splitting. A fixed seed does not make every hardware or library implementation identical, but it makes this notebook's sampling decisions repeatable and auditable. Reproducibility matters because a later training result is only meaningful if the data split can be recreated.

**GPU note:** a GPU is not required for this notebook. Dataset loading, inspection, sampling, statistics, plotting, and basic tokenizer measurements can all be performed on CPU. We should not allocate a GPU merely for exploration.

In [ ]:
import json
import platform
import random
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from datasets import ClassLabel, Dataset, load_dataset

sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def is_missing_value(value):
    """Return True for scalar null-like values without treating arrays as booleans."""
    if value is None:
        return True
    try:
        result = pd.isna(value)
    except (TypeError, ValueError):
        return False
    if isinstance(result, (bool, np.bool_)):
        return bool(result)
    return False


def category_label(value):
    """Make task labels safe to display and use as deterministic table indexes."""
    return "<MISSING>" if is_missing_value(value) else str(value)


def preview_text(value, limit=500):
    """Keep notebook output readable while leaving the underlying value unchanged."""
    if is_missing_value(value):
        return "<MISSING>"
    text = str(value).replace("\r\n", "\n")
    return text if len(text) <= limit else text[:limit].rstrip() + " …"


def as_canonical_value(value):
    """Convert scalar source values to text but preserve missingness for final assertions."""
    return None if is_missing_value(value) else str(value)


try:
    import torch

    torch_installed = True
    cuda_available = bool(torch.cuda.is_available())
except ImportError:
    torch = None
    torch_installed = False
    cuda_available = False

print("Python version:", platform.python_version())
print("pandas version:", pd.__version__)
print("datasets version:", __import__("datasets").__version__)
print("PyTorch installed:", torch_installed)
if torch_installed:
    print("PyTorch version:", torch.__version__)
print("CUDA available:", cuda_available)
if cuda_available:
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU name: not available — CPU execution is sufficient for this notebook")
print("Random seed:", SEED)

The environment report is intentionally informative rather than prescriptive. A `False` CUDA value is expected to be acceptable: this notebook does not perform gradient computation or model training.

## 3. Load the Hugging Face dataset

We load the `train` split as a Hugging Face `Dataset` object. Inspecting that object first avoids the unnecessary memory cost of converting all approximately 158,000 rows and all long clinical notes into a pandas DataFrame. We will materialize only small tables or the selected 5,000 examples when that is useful.

In [ ]:
DATASET_NAME = "aisc-team-a1/Asclepius-Synthetic-Clinical-Notes"
DATASET_SOURCE = "https://huggingface.co/datasets/aisc-team-a1/Asclepius-Synthetic-Clinical-Notes"

raw = load_dataset(DATASET_NAME, split="train")

print("Dataset object:", raw)
print("Number of rows:", len(raw))
print("Column names:", raw.column_names)
print("\nFeature/schema information:")
display(raw.features)

print("\nA few raw records (text is previewed only for readable output):")
for row_index in range(min(3, len(raw))):
    row = raw[row_index]
    print(f"\n--- raw row {row_index} ---")
    for column_name in raw.column_names:
        print(f"{column_name}: {preview_text(row[column_name], limit=350)}")

The output above is the first source of truth. The next cell verifies the fields needed for supervised fine-tuning instead of assuming that the expected names are present. The case identifier is resolved from a short list of likely names; if no such field exists, the notebook stops because case-level leakage checks would not be defensible.

In [ ]:
# Required fields are verified against the loaded schema; they are not added by assumption.
required_text_candidates = {
    "note": ["note", "clinical_note", "text"],
    "question": ["question", "instruction", "prompt"],
    "answer": ["answer", "target", "response"],
    "task": ["task", "task_type", "category"],
}

resolved_columns = {}
for role, candidates in required_text_candidates.items():
    resolved_columns[role] = next(
        (candidate for candidate in candidates if candidate in raw.column_names), None
    )

missing_roles = [role for role, column in resolved_columns.items() if column is None]
if missing_roles:
    raise ValueError(
        f"The loaded schema is missing required roles: {missing_roles}. "
        f"Available columns are: {raw.column_names}"
    )

case_candidates = ["patient_id", "case_id", "patient", "case", "encounter_id", "id"]
case_column = next((candidate for candidate in case_candidates if candidate in raw.column_names), None)
if case_column is None:
    raise ValueError(
        "No patient/case identifier was found. Stop rather than creating a random "
        "row split that could leak clinical cases."
    )

note_column = resolved_columns["note"]
question_column = resolved_columns["question"]
answer_column = resolved_columns["answer"]
task_column = resolved_columns["task"]

print("Verified source roles:")
print("  clinical note:", note_column)
print("  instruction/question:", question_column)
print("  target/answer:", answer_column)
print("  task:", task_column)
print("  case/patient identifier:", case_column)

schema_table = pd.DataFrame(
    {
        "Role": ["clinical note", "instruction/question", "target/answer", "task", "case/patient"],
        "Actual column": [note_column, question_column, answer_column, task_column, case_column],
        "Feature": [str(raw.features[column]) for column in [note_column, question_column, answer_column, task_column, case_column]],
    }
)
display(schema_table)

## 4. Basic data-quality inspection

This scan checks every source column for scalar null-like values and empty strings. It also checks the required text roles for obvious malformed values: missing, empty, or non-string content. The scan works in batches and does not convert the full dataset into a pandas table.

No values are changed in this section. A malformed required field is a reason to investigate or fail loudly, not a reason to silently coerce it away.

In [ ]:
BATCH_SIZE = 2_048
quality_rows = []
example_values = {}
malformed_counts = {role: 0 for role in required_text_candidates}

for start in range(0, len(raw), BATCH_SIZE):
    stop = min(start + BATCH_SIZE, len(raw))
    batch = raw[start:stop]

    for column_name in raw.column_names:
        values = batch[column_name]
        missing_count = sum(is_missing_value(value) for value in values)
        empty_count = sum(
            isinstance(value, str) and not value.strip() for value in values
        )
        for value in values:
            if column_name not in example_values and not is_missing_value(value):
                example_values[column_name] = value

        quality_rows.append(
            {
                "Column": column_name,
                "Type": str(raw.features[column_name]),
                "Missing": int(missing_count),
                "Empty": int(empty_count),
                "Example": preview_text(example_values.get(column_name), limit=180),
            }
        )

    # Notes, questions, and answers must be nonempty strings. A task can be a
    # valid integer-backed Hugging Face ClassLabel, so task labels are checked
    # for null/empty values without requiring the raw storage type to be str.
    for role in ["note", "question", "answer"]:
        column_name = resolved_columns[role]
        for value in batch[column_name]:
            if is_missing_value(value) or not isinstance(value, str) or not value.strip():
                malformed_counts[role] += 1
    task_source_column = resolved_columns["task"]
    for value in batch[task_source_column]:
        if is_missing_value(value) or (isinstance(value, str) and not value.strip()):
            malformed_counts["task"] += 1

# Each source column was seen once per batch; aggregate the batch-level rows.
quality_table = (
    pd.DataFrame(quality_rows)
    .groupby(["Column", "Type"], as_index=False)[["Missing", "Empty"]]
    .sum()
)
quality_table["Example"] = quality_table["Column"].map(
    lambda column_name: preview_text(example_values.get(column_name), limit=180)
)
quality_table = quality_table[["Column", "Type", "Missing", "Empty", "Example"]]
print("Null/empty summary for every source column:")
display(quality_table)

malformed_table = pd.DataFrame(
    [{"Role": role, "Column": resolved_columns[role], "Malformed rows": count}
     for role, count in malformed_counts.items()]
)
print("\nObvious malformed values in required roles:")
display(malformed_table)

if any(malformed_counts.values()):
    raise ValueError(
        "At least one required role contains a missing, empty, or non-string value. "
        "Review the quality tables before continuing; the notebook will not silently repair it."
    )

**Interpretation.** A clean table has zero missing and empty values for the fields used as note, instruction, answer, and task. The case identifier is also checked separately below. If a nonzero count appears, stop and investigate the corresponding rows before sampling; dropping bad rows without documenting it would change the experiment.

## 5. Inspect representative examples from different tasks

The task label tells us what kind of supervised behavior an example represents, but it does not tell us whether the note, instruction, and answer are actually arranged as expected. We therefore show at most one representative row for each of the first eight distinct task labels encountered. Long text is previewed only to keep the notebook readable; the dataset values are not truncated in memory or on disk.

In [ ]:
representative_indices = {}

for start in range(0, len(raw), BATCH_SIZE):
    stop = min(start + BATCH_SIZE, len(raw))
    batch = raw[start:stop]
    for offset, task_value in enumerate(batch[task_column]):
        label = category_label(task_value)
        representative_indices.setdefault(label, start + offset)
        if len(representative_indices) >= 8:
            break
    if len(representative_indices) >= 8:
        break

print(f"Showing {len(representative_indices)} representative task examples:")
for label, row_index in representative_indices.items():
    row = raw[int(row_index)]
    print(f"\n{'=' * 80}\nTASK: {label}\nPATIENT/CASE ID: {row[case_column]}")
    print("NOTE:\n", preview_text(row[note_column], limit=1_200))
    print("QUESTION / INSTRUCTION:\n", preview_text(row[question_column], limit=900))
    print("ANSWER:\n", preview_text(row[answer_column], limit=900))

**Interpretation.** Read the examples as input/target pairs, not as ordinary classification rows: the note and question provide the input context, while the answer is the desired clinical response. The examples also tell us whether a task label is a meaningful grouping variable for stratified sampling.

## 6. Task distribution

We now count every task and calculate its percentage of the full source. This analysis happens before the 5,000-example selection so that the selection rule can preserve the observed distribution rather than accidentally inheriting the order of the source file.

In [ ]:
task_feature = raw.features[task_column]
if isinstance(task_feature, ClassLabel):
    task_labels = [
        "<MISSING>" if is_missing_value(value) else task_feature.int2str(int(value))
        for value in raw[task_column]
    ]
else:
    task_labels = [category_label(value) for value in raw[task_column]]
full_task_counts = pd.Series(task_labels, name="Count").value_counts(sort=False)
full_task_percentages = 100 * full_task_counts / len(raw)

task_distribution = pd.DataFrame(
    {
        "Task": full_task_counts.index.tolist(),
        "Count": full_task_counts.astype(int).tolist(),
        "Percentage": full_task_percentages.round(3).tolist(),
    }
)
print("Full dataset task distribution:")
display(task_distribution)

fig, ax = plt.subplots(figsize=(11, 4.5))
sns.barplot(data=task_distribution, x="Task", y="Count", color="steelblue", ax=ax)
ax.set_title("Asclepius task distribution in the full dataset")
ax.set_xlabel("Task")
ax.set_ylabel("Examples")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

**Interpretation.** Compare the percentages rather than only the raw counts. A heavily skewed source can make a random 5,000-row sample underrepresent smaller tasks; a seeded task-stratified sample gives each task a controlled share while still reflecting the source distribution.

## 7. Investigate patient/case grouping

A case can have more than one supervised example. If rows from the same case are randomly split, the model may see nearly identical clinical context during training and evaluation, making the evaluation look better than it is.

The safe pattern is:

```text
clinical cases
      ↓
split by patient/case identifier
      ↓
train cases    validation cases    test cases
```

The unsafe pattern is:

```text
same clinical case → train row + test row
```

The following metadata table contains only row index, task label, and case identifier—not the long note, question, or answer—so it remains inexpensive compared with a full text DataFrame.

In [ ]:
row_metadata = pd.DataFrame(
    {
        "row_index": np.arange(len(raw), dtype=np.int64),
        "case_id_value": raw[case_column],
        "task": task_labels,
    }
)

missing_case_rows = int(row_metadata["case_id_value"].map(is_missing_value).sum())
print("Case identifier column:", case_column)
print("Rows with missing case identifier:", missing_case_rows)
if missing_case_rows:
    raise ValueError(
        "Missing case identifiers prevent a defensible case-aware split. "
        "Stop and repair the source data rather than introducing leakage."
    )

case_counts = row_metadata.groupby("case_id_value", sort=False).size()
number_of_cases = int(case_counts.size)
duplicate_case_count = int((case_counts > 1).sum())
rows_in_duplicate_cases = int(case_counts[case_counts > 1].sum())

case_size_summary = pd.DataFrame(
    {
        "Statistic": ["Unique cases", "Rows", "Minimum rows/case", "Median rows/case", "Mean rows/case", "Maximum rows/case", "Cases with >1 row", "Rows in multi-row cases"],
        "Value": [
            number_of_cases,
            len(raw),
            int(case_counts.min()),
            float(case_counts.median()),
            float(case_counts.mean()),
            int(case_counts.max()),
            duplicate_case_count,
            rows_in_duplicate_cases,
        ],
    }
)
display(case_size_summary)

print("\nDistribution of examples per case (selected quantiles):")
display(case_counts.describe(percentiles=[0.50, 0.90, 0.95, 0.99]).to_frame("Rows per case"))

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(case_counts, discrete=True, color="darkorange", ax=ax)
ax.set_title("Examples per patient/case")
ax.set_xlabel("Number of examples belonging to one case")
ax.set_ylabel("Number of cases")
plt.tight_layout()
plt.show()

has_duplicate_cases = bool((case_counts > 1).any())
print("Multiple rows per case detected:", has_duplicate_cases)

**Interpretation.** If every case has one row, a row split is also case-safe because no case can occur in two rows. If any case has multiple rows, all rows for a chosen case must travel together during both selection and splitting. The next section has an explicit branch for those two situations and fails if an exact requested size cannot be achieved with whole cases.

## 8. Select a controlled 5,000-example subset

The target is exactly 5,000 examples. We will not take the first 5,000 rows. Instead, we first compute proportional task quotas using the largest-remainder method so the integer quotas sum exactly to 5,000.

- With unique case identifiers, rows can be sampled within each task using the fixed seed.
- With duplicate identifiers, whole cases are sampled. If every case contains one task, the same task quotas are enforced at the case level when an exact whole-case solution exists.
- If a per-task quota is not representable by whole cases, or if cases contain multiple tasks, we preserve the exact total size, report the resulting distribution, and do not split a case.
- If whole cases cannot sum to exactly 5,000, the notebook raises an error instead of silently taking part of a case.

In [ ]:
TARGET_SIZE = 5_000
if len(raw) < TARGET_SIZE:
    raise ValueError(f"The source has only {len(raw):,} rows; cannot select {TARGET_SIZE:,} examples.")


def allocate_counts_from_proportions(counts, total):
    """Allocate an integer total proportionally, resolving rounding by largest remainder."""
    counts = counts.astype(float)
    if total < 0 or total > counts.sum():
        raise ValueError("Requested allocation is outside the available count.")
    if total == 0:
        return pd.Series(0, index=counts.index, dtype="int64")
    ideal = counts / counts.sum() * total
    allocated = np.floor(ideal).astype("int64")
    leftover = int(total - allocated.sum())
    fractional_part = (ideal - allocated).sort_values(ascending=False, kind="stable")
    for label in fractional_part.index[:leftover]:
        allocated.loc[label] += 1
    assert int(allocated.sum()) == total
    return allocated.astype("int64")


desired_task_counts = allocate_counts_from_proportions(full_task_counts, TARGET_SIZE)
desired_quota_table = pd.DataFrame(
    {
        "Task": full_task_counts.index.tolist(),
        "Full Count": full_task_counts.astype(int).tolist(),
        "Full %": (100 * full_task_counts / len(raw)).round(3).tolist(),
        "Target selected count": desired_task_counts.astype(int).tolist(),
    }
)
display(desired_quota_table)
print("Target quota total:", int(desired_task_counts.sum()))

The largest-remainder calculation is deterministic: the full-data proportions determine the base quotas, and the few rounding leftovers go to the tasks with the largest fractional remainders. This keeps the 5,000-example target exact without privileging the source row order.

### Whole-case selection helper

When a case has several rows, an exact 5,000-row target may require choosing a combination of whole cases whose group sizes add to 5,000. The bounded subset-sum helper below works only on the small case-size metadata table; it never loads or manipulates clinical text. If no exact combination exists, it returns `None` so the notebook can stop explicitly.

In [ ]:
def exact_group_size_counts(group_sizes, target):
    """Find how many groups of each size add up to target, or return None."""
    sizes = pd.Series(group_sizes, dtype="int64")
    if target == 0:
        return {}

    # Binary chunks turn a bounded count of equal-sized groups into a small
    # 0/1 subset-sum problem. The target is at most 5,000 in this notebook.
    available = sizes.value_counts().sort_index()
    chunks = []
    for size_value, count_value in available.items():
        size = int(size_value)
        remaining = int(count_value)
        power = 1
        while remaining > 0:
            take = min(power, remaining)
            weight = size * take
            if weight <= target:
                chunks.append((size, take, weight))
            remaining -= take
            power *= 2

    reachable = [False] * (target + 1)
    parent_sum = [-1] * (target + 1)
    parent_size = [-1] * (target + 1)
    parent_take = [-1] * (target + 1)
    reachable[0] = True

    # Descending totals prevent a chunk from being reused in the same pass.
    for size, take, weight in chunks:
        for subtotal in range(target - weight, -1, -1):
            new_total = subtotal + weight
            if reachable[subtotal] and not reachable[new_total]:
                reachable[new_total] = True
                parent_sum[new_total] = subtotal
                parent_size[new_total] = size
                parent_take[new_total] = take

    if not reachable[target]:
        return None

    chosen_counts = {}
    current = target
    while current > 0:
        previous = parent_sum[current]
        if previous < 0:
            raise RuntimeError("Subset-sum reconstruction failed unexpectedly.")
        size = parent_size[current]
        chosen_counts[size] = chosen_counts.get(size, 0) + parent_take[current]
        current = previous
    return chosen_counts

In [ ]:
def choose_case_ids_exact_rows(case_sizes, target, seed):
    """Sample whole case IDs whose row counts add up to target, or return None."""
    counts_by_size = exact_group_size_counts(case_sizes, target)
    if counts_by_size is None:
        return None

    chosen_case_ids = []
    for offset, (size, number_to_take) in enumerate(sorted(counts_by_size.items())):
        candidates = case_sizes[case_sizes == size].index.tolist()
        if number_to_take > len(candidates):
            raise RuntimeError("The requested group-size combination exceeds available case IDs.")
        sampled = pd.Series(candidates).sample(
            n=int(number_to_take), random_state=seed + offset
        ).tolist()
        chosen_case_ids.extend(sampled)
    return chosen_case_ids


case_table = (
    row_metadata.groupby("case_id_value", sort=False, as_index=False)
    .agg(row_count=("row_index", "size"), task_count=("task", "nunique"))
)
case_first_task = row_metadata.groupby("case_id_value", sort=False)["task"].first().rename("task")
case_table = case_table.merge(case_first_task, on="case_id_value", how="left")

display(case_table.head())

The case table makes the constraint visible: `row_count` is the number of examples that must move together, and `task_count` tells us whether a case mixes tasks. We now choose rows for the unique-case branch or whole cases for the duplicate-case branch.

In [ ]:
if not has_duplicate_cases:
    # Case-safe because each selected row represents a different case.
    selected_indices = []
    for position, task in enumerate(full_task_counts.index):
        candidates = row_metadata.loc[row_metadata["task"] == task, "row_index"].tolist()
        number_to_take = int(desired_task_counts.loc[task])
        sampled = pd.Series(candidates).sample(
            n=number_to_take, random_state=SEED + position
        ).tolist()
        selected_indices.extend(int(index) for index in sampled)
    selection_method = "seeded task-stratified row sampling; case identifiers are unique"
else:
    case_task_is_single = bool((case_table["task_count"] == 1).all())
    if case_task_is_single:
        task_selected_case_ids = []
        quotas_representable = True
        for position, task in enumerate(full_task_counts.index):
            task_case_sizes = case_table.loc[
                case_table["task"] == task, ["case_id_value", "row_count"]
            ].set_index("case_id_value")["row_count"]
            chosen = choose_case_ids_exact_rows(
                task_case_sizes, int(desired_task_counts.loc[task]), SEED + position
            )
            if chosen is None:
                quotas_representable = False
                break
            task_selected_case_ids.extend(chosen)

        if quotas_representable:
            selected_case_ids = task_selected_case_ids
            selection_method = "seeded task-stratified whole-case sampling with exact quotas"
        else:
            # Preserve the more important exact total/case constraint when an
            # individual task quota cannot be represented by whole groups.
            all_case_sizes = case_table.set_index("case_id_value")["row_count"]
            selected_case_ids = choose_case_ids_exact_rows(all_case_sizes, TARGET_SIZE, SEED)
            if selected_case_ids is None:
                raise ValueError(
                    "The case groups cannot sum to exactly 5,000 rows. Stop rather than "
                    "breaking cases or silently changing the requested dataset size."
                )
            selection_method = (
                "seeded whole-case sampling with exact total size; per-task quotas "
                "were not exactly representable by whole cases"
            )
    else:
        all_case_sizes = case_table.set_index("case_id_value")["row_count"]
        selected_case_ids = choose_case_ids_exact_rows(all_case_sizes, TARGET_SIZE, SEED)
        if selected_case_ids is None:
            raise ValueError(
                "The case groups cannot sum to exactly 5,000 rows. Stop rather than "
                "breaking cases or silently changing the requested dataset size."
            )
        selection_method = (
            "seeded whole-case sampling with exact total size; mixed-task cases "
            "take precedence over exact task quotas"
        )

    selected_indices = row_metadata.loc[
        row_metadata["case_id_value"].isin(selected_case_ids), "row_index"
    ].astype(int).tolist()

In [ ]:
selected_indices = [int(index) for index in selected_indices]
if len(set(selected_indices)) != len(selected_indices):
    raise RuntimeError("The selection unexpectedly contains duplicate row indices.")
if len(selected_indices) != TARGET_SIZE:
    raise RuntimeError(
        f"Selection produced {len(selected_indices):,} rows instead of {TARGET_SIZE:,}."
    )

selected_metadata = row_metadata[
    row_metadata["row_index"].isin(selected_indices)
].copy()
selected_raw = raw.select(selected_indices)
selected_task_counts = (
    selected_metadata["task"].value_counts(sort=False)
    .reindex(full_task_counts.index, fill_value=0)
    .astype("int64")
)
selected_task_percentages = 100 * selected_task_counts / TARGET_SIZE

selection_comparison = pd.DataFrame(
    {
        "Task": full_task_counts.index.tolist(),
        "Full Count": full_task_counts.astype(int).tolist(),
        "Full %": (100 * full_task_counts / len(raw)).round(3).tolist(),
        "Selected Count": selected_task_counts.astype(int).tolist(),
        "Selected %": selected_task_percentages.round(3).tolist(),
    }
)
print("Full dataset distribution ↓")
display(task_distribution)
print("Selected 5,000 distribution ↓")
display(selection_comparison)
print("\nSelection method:", selection_method)
print("Selected rows:", len(selected_raw))
print("Selected unique cases:", selected_metadata["case_id_value"].nunique())
print(
    "Largest absolute task-percentage difference (percentage points):",
    round(float((selected_task_percentages - full_task_percentages).abs().max()), 3),
)

**Interpretation.** The comparison table is the audit trail for the selection decision. With unique cases or compatible whole-case sizes, the selected counts follow the source quotas closely (and the quota construction is exact). With mixed-task cases, a small task deviation can be the correct result because avoiding case fragmentation is more important than forcing a row-level quota. The code reports that trade-off rather than hiding it.

## 9. Create the 4,000 / 500 / 500 split

The 5,000 examples are split only after selection. The requested sizes are:

- **Train:** 4,000 examples (80%)
- **Validation:** 500 examples (10%)
- **Test:** 500 examples (10%)

When cases are unique, we stratify each split by task with deterministic integer quotas. When cases have multiple rows, we select whole cases for validation and test first, then assign the remaining cases to train. In both branches, the final overlap checks below are mandatory.

In [ ]:
TRAIN_SIZE = 4_000
VALIDATION_SIZE = 500
TEST_SIZE = 500
assert TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE == TARGET_SIZE

selected_has_duplicate_cases = bool(
    selected_metadata.groupby("case_id_value", sort=False).size().gt(1).any()
)

if not selected_has_duplicate_cases:
    # Allocate train first, then validation from the remaining rows; test receives the rest.
    train_task_counts = allocate_counts_from_proportions(selected_task_counts, TRAIN_SIZE)
    remaining_after_train = selected_task_counts - train_task_counts
    validation_task_counts = allocate_counts_from_proportions(
        remaining_after_train, VALIDATION_SIZE
    )
    test_task_counts = selected_task_counts - train_task_counts - validation_task_counts

    train_row_indices = []
    validation_row_indices = []
    test_row_indices = []
    for position, task in enumerate(full_task_counts.index):
        candidates = selected_metadata.loc[
            selected_metadata["task"] == task, "row_index"
        ].astype(int).tolist()
        shuffled = pd.Series(candidates).sample(
            frac=1, random_state=SEED + 1_000 + position
        ).tolist()
        train_n = int(train_task_counts.loc[task])
        validation_n = int(validation_task_counts.loc[task])
        train_row_indices.extend(shuffled[:train_n])
        validation_row_indices.extend(shuffled[train_n:train_n + validation_n])
        test_row_indices.extend(shuffled[train_n + validation_n:])

    split_method = "seeded task-stratified row split; unique case IDs make it case-safe"
else:
    selected_case_sizes = selected_metadata.groupby("case_id_value", sort=False).size()
    validation_case_ids = choose_case_ids_exact_rows(
        selected_case_sizes, VALIDATION_SIZE, SEED + 2_000
    )
    if validation_case_ids is None:
        raise ValueError(
            "Selected whole cases cannot form exactly 500 validation rows. "
            "Stop rather than splitting a case."
        )

    remaining_case_sizes = selected_case_sizes.drop(index=validation_case_ids)
    test_case_ids = choose_case_ids_exact_rows(
        remaining_case_sizes, TEST_SIZE, SEED + 3_000
    )
    if test_case_ids is None:
        raise ValueError(
            "The remaining whole cases cannot form exactly 500 test rows. "
            "Stop rather than splitting a case."
        )

    train_case_ids = remaining_case_sizes.drop(index=test_case_ids).index.tolist()

    def row_indices_for_case_ids(case_ids):
        return selected_metadata.loc[
            selected_metadata["case_id_value"].isin(case_ids), "row_index"
        ].astype(int).tolist()

    validation_row_indices = row_indices_for_case_ids(validation_case_ids)
    test_row_indices = row_indices_for_case_ids(test_case_ids)
    train_row_indices = row_indices_for_case_ids(train_case_ids)
    split_method = "seeded whole-case split; task proportions are reported but case integrity takes precedence"

print("Split strategy chosen:", split_method)

The split-membership code above chooses either task-stratified rows (when every case is unique) or whole cases (when a case owns multiple rows). The next cell only shuffles split order, materializes the three source datasets, and performs the leakage assertions.

In [ ]:
# Shuffle the order inside each saved split without changing membership.
train_row_indices = pd.Series(train_row_indices).sample(
    frac=1, random_state=SEED + 4_000
).astype(int).tolist()
validation_row_indices = pd.Series(validation_row_indices).sample(
    frac=1, random_state=SEED + 5_000
).astype(int).tolist()
test_row_indices = pd.Series(test_row_indices).sample(
    frac=1, random_state=SEED + 6_000
).astype(int).tolist()

print("Train rows:", len(train_row_indices))
print("Validation rows:", len(validation_row_indices))
print("Test rows:", len(test_row_indices))

# Build the three source Dataset objects while case IDs are still available for auditing.
train_raw = raw.select(train_row_indices)
validation_raw = raw.select(validation_row_indices)
test_raw = raw.select(test_row_indices)


def case_ids_for_rows(row_indices):
    return set(
        row_metadata.loc[
            row_metadata["row_index"].isin(row_indices), "case_id_value"
        ].tolist()
    )

train_case_ids = case_ids_for_rows(train_row_indices)
validation_case_ids = case_ids_for_rows(validation_row_indices)
test_case_ids = case_ids_for_rows(test_row_indices)

print("\nExplicit leakage checks:")
print("Train/Test case overlap:", len(train_case_ids & test_case_ids))
print("Train/Validation case overlap:", len(train_case_ids & validation_case_ids))
print("Validation/Test case overlap:", len(validation_case_ids & test_case_ids))

assert len(train_raw) == TRAIN_SIZE
assert len(validation_raw) == VALIDATION_SIZE
assert len(test_raw) == TEST_SIZE
assert len(train_case_ids & test_case_ids) == 0
assert len(train_case_ids & validation_case_ids) == 0
assert len(validation_case_ids & test_case_ids) == 0

**Interpretation.** The three overlap values must all be zero. The row counts must be exactly 4,000, 500, and 500. A test set with a nonzero case overlap is not an acceptable evaluation set, even if its row count is correct.

## 10. Analyze note, question, and answer lengths

We measure character and whitespace-delimited word counts before deciding anything about tokenization or truncation. Long examples are not discarded here. These distributions tell the next notebook how much variation exists and where a tokenizer/context-window analysis is needed.

In [ ]:
length_sources = {
    "note": note_column,
    "question": question_column,
    "answer": answer_column,
}
length_data = {}

for label, source_column in length_sources.items():
    text_values = [as_canonical_value(value) or "" for value in selected_raw[source_column]]
    length_data[f"{label}_characters"] = [len(text) for text in text_values]
    length_data[f"{label}_words"] = [len(text.split()) for text in text_values]

text_lengths = pd.DataFrame(length_data)
length_summary_rows = []
for column_name in text_lengths.columns:
    series = text_lengths[column_name]
    field, unit = column_name.rsplit("_", 1)
    length_summary_rows.append(
        {
            "Field": field,
            "Measure": unit,
            "Mean": round(float(series.mean()), 2),
            "Median": round(float(series.median()), 2),
            "Minimum": int(series.min()),
            "Maximum": int(series.max()),
            "P90": round(float(series.quantile(0.90)), 2),
            "P95": round(float(series.quantile(0.95)), 2),
            "P99": round(float(series.quantile(0.99)), 2),
        }
    )
length_summary = pd.DataFrame(length_summary_rows)
print("Character and whitespace-word statistics for the selected 5,000 examples:")
display(length_summary)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for column_name, ax in zip(text_lengths.columns, axes.flat):
    sns.histplot(text_lengths[column_name], bins=40, color="slateblue", ax=ax)
    ax.set_title(column_name.replace("_", " ").title())
    ax.set_xlabel("Count")
    ax.set_ylabel("Examples")
plt.tight_layout()
plt.show()

**Interpretation.** Character and word counts are useful sanity checks, but model limits are measured in tokens, not words. The maximum and upper percentiles identify potentially long-tail examples. We keep them in the saved data and defer any truncation decision until the exact base model and its tokenizer are selected.

## 11. Create the canonical supervised-learning representation

Before selecting a model-specific chat template, we make the data model-independent. Every row becomes:

```python
{
    "task": "...",
    "clinical_note": "...",
    "instruction": "...",
    "target": "...",
}
```

The note and instruction are the input. The answer is the target. We retain case IDs only in temporary audit metadata; they are not needed by the next notebook's training loader and are not included in the canonical saved columns.

This separation matters because different LLMs use different chat templates, role names, separators, and special tokens. The next notebook will apply the selected model's actual template rather than hardcoding one here.

In [ ]:
task_feature = raw.features[task_column]

def canonical_task_value(value):
    if is_missing_value(value):
        return None
    if isinstance(task_feature, ClassLabel):
        return task_feature.int2str(int(value))
    return str(value)


def make_canonical_dataset(source_dataset):
    # Explicit columns make the representation easy to inspect and avoid carrying
    # source-specific fields into the next notebook.
    return Dataset.from_dict(
        {
            "task": [canonical_task_value(value) for value in source_dataset[task_column]],
            "clinical_note": [as_canonical_value(value) for value in source_dataset[note_column]],
            "instruction": [as_canonical_value(value) for value in source_dataset[question_column]],
            "target": [as_canonical_value(value) for value in source_dataset[answer_column]],
        }
    )


train_canonical = make_canonical_dataset(train_raw)
validation_canonical = make_canonical_dataset(validation_raw)
test_canonical = make_canonical_dataset(test_raw)
canonical_datasets = {
    "train": train_canonical,
    "validation": validation_canonical,
    "test": test_canonical,
}

print("Canonical columns:", train_canonical.column_names)
print("Canonical features:")
display(train_canonical.features)
print("Canonical training rows:", len(train_canonical))

## 12. Inspect formatted supervised examples and run structure checks

The examples below are shown in the form a supervised fine-tuning pipeline will consume:

```text
INPUT  = clinical note + instruction
TARGET = desired clinical response
```

This is still not a chat template and it is not tokenized. We check that the source fields are nonempty and report exact target/source substring overlap for manual review. Some legitimate clinical answers can repeat a phrase from a note, so overlap is a diagnostic rather than an automatic deletion rule.

In [ ]:
print("Canonical supervised examples from the training split:")
for example_index in range(min(8, len(train_canonical))):
    example = train_canonical[example_index]
    print(f"\n{'=' * 80}\nEXAMPLE {example_index + 1}\nTASK: {example['task']}")
    print("INPUT — CLINICAL NOTE:\n", preview_text(example["clinical_note"], limit=1_000))
    print("INPUT — INSTRUCTION:\n", preview_text(example["instruction"], limit=700))
    print("TARGET — DESIRED RESPONSE:\n", preview_text(example["target"], limit=700))

In [ ]:
format_check_rows = []
for split_name, split_dataset in canonical_datasets.items():
    empty_note = 0
    empty_instruction = 0
    empty_target = 0
    target_in_input = 0
    target_equal_to_input = 0

    for example in split_dataset:
        note = example["clinical_note"]
        instruction = example["instruction"]
        target = example["target"]
        note_text = "" if is_missing_value(note) else str(note)
        instruction_text = "" if is_missing_value(instruction) else str(instruction)
        target_text = "" if is_missing_value(target) else str(target)
        input_text = f"{note_text.strip()}\n\n{instruction_text.strip()}".strip()

        empty_note += not note_text.strip()
        empty_instruction += not instruction_text.strip()
        empty_target += not target_text.strip()
        target_in_input += bool(target_text.strip() and target_text.strip() in input_text)
        target_equal_to_input += bool(target_text.strip() and target_text.strip() == input_text)

    format_check_rows.append(
        {
            "Split": split_name,
            "Rows": len(split_dataset),
            "Empty clinical notes": int(empty_note),
            "Empty instructions": int(empty_instruction),
            "Empty targets": int(empty_target),
            "Target appears verbatim in input": int(target_in_input),
            "Target equals complete input": int(target_equal_to_input),
        }
    )

format_checks = pd.DataFrame(format_check_rows)
display(format_checks)
print(
    "Target-in-input rows are reported for review; they are not removed automatically, "
    "because legitimate answers can quote a source phrase. The canonical construction "
    "never concatenates the target into the input."
)

**Interpretation.** The important structural checks are zero empty instructions and targets, and zero rows where the target equals the complete input. If a target appears as a substring, inspect representative rows rather than assuming leakage: clinical answers can legitimately restate facts from the note. No chat-specific markers or model special tokens have been introduced.

## 13. Optional tokenizer compatibility analysis

The exact 3B base model has not been selected in this notebook, so we do **not** guess a tokenizer, a chat template, or a context length. Leave `TOKENIZER_NAME = None` to defer this work. If a tokenizer is already known, set that variable to its Hugging Face identifier and rerun this section; `AutoTokenizer` will load the tokenizer without loading model weights.

When the model is selected later, the relevant quantity is:

```text
input tokens + target tokens + chat/special tokens
```

We calculate lengths without truncation. No `max_length = 4096` (or any other guessed value) is used here.

In [ ]:
# Keep this None until the final base model is chosen in Notebook 02.
TOKENIZER_NAME = None
TOKENIZER_CONTEXT_LENGTH = None
tokenizer_analysis_completed = False

tokenizer_summary = None

if TOKENIZER_NAME is None:
    print(
        "Tokenizer analysis deferred: select the exact 3B base model in the next notebook "
        "before choosing a tokenizer or context window."
    )
else:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)
    print("Tokenizer:", TOKENIZER_NAME)
    print("Tokenizer class:", tokenizer.__class__.__name__)
    print("Tokenizer-reported model_max_length:", tokenizer.model_max_length)

    # A very large sentinel generally means that the tokenizer does not know the
    # model context window. We do not turn that sentinel into a guessed limit.
    reported_context = tokenizer.model_max_length
    if reported_context is not None and int(reported_context) < 1_000_000:
        TOKENIZER_CONTEXT_LENGTH = int(reported_context)
        print("Usable tokenizer-reported context length:", TOKENIZER_CONTEXT_LENGTH)
    else:
        print("Context length is not available from the tokenizer; check the model config later.")

    token_length_rows = []
    for split_name, split_dataset in canonical_datasets.items():
        for example in split_dataset:
            input_text = (
                f"{example['clinical_note'].strip()}\n\n{example['instruction'].strip()}"
            )
            input_token_count = len(
                tokenizer(input_text, add_special_tokens=True, truncation=False)["input_ids"]
            )
            target_token_count = len(
                tokenizer(example["target"], add_special_tokens=False, truncation=False)["input_ids"]
            )
            token_length_rows.append(
                {
                    "split": split_name,
                    "input_tokens": input_token_count,
                    "target_tokens": target_token_count,
                    "input_plus_target_tokens": input_token_count + target_token_count,
                }
            )

    token_lengths = pd.DataFrame(token_length_rows)
    token_summary_rows = []
    for column_name in ["input_tokens", "target_tokens", "input_plus_target_tokens"]:
        series = token_lengths[column_name]
        token_summary_rows.append(
            {
                "Measure": column_name,
                "Median": float(series.median()),
                "P90": float(series.quantile(0.90)),
                "P95": float(series.quantile(0.95)),
                "P99": float(series.quantile(0.99)),
                "Maximum": int(series.max()),
            }
        )
    tokenizer_summary = pd.DataFrame(token_summary_rows)
    tokenizer_analysis_completed = True
    display(tokenizer_summary)

    if TOKENIZER_CONTEXT_LENGTH is not None:
        over_context = token_lengths["input_plus_target_tokens"] > TOKENIZER_CONTEXT_LENGTH
        print(
            "Percentage exceeding the tokenizer-reported context window:",
            round(100 * float(over_context.mean()), 3),
        )
    else:
        print("Percentage exceeding context window: not computed because the context is unknown.")

**Interpretation.** The default output should say that tokenizer analysis is deferred. That is intentional: context length is a property of the selected model/tokenizer and must not be guessed. The next notebook will apply the model's actual chat template and repeat the measurement, including its special tokens, before setting a training length or deciding how to handle long examples.

## 14. Final dataset validation

The following assertions are deliberately explicit. A failure should stop the notebook and identify a data or split problem before anything is saved.

In [ ]:
# Exact requested sizes.
assert len(train_canonical) == TRAIN_SIZE, len(train_canonical)
assert len(validation_canonical) == VALIDATION_SIZE, len(validation_canonical)
assert len(test_canonical) == TEST_SIZE, len(test_canonical)
assert len(train_canonical) + len(validation_canonical) + len(test_canonical) == TARGET_SIZE

# Required canonical fields must be present and nonempty.
canonical_required_fields = ["task", "clinical_note", "instruction", "target"]
for split_name, split_dataset in canonical_datasets.items():
    assert split_dataset.column_names == canonical_required_fields
    for field in canonical_required_fields:
        bad_indices = [
            index
            for index, value in enumerate(split_dataset[field])
            if is_missing_value(value) or not str(value).strip()
        ]
        assert not bad_indices, f"{split_name}.{field} has invalid rows: {bad_indices[:10]}"

# Row membership and case membership must both be disjoint.
assert len(set(train_row_indices) & set(validation_row_indices)) == 0
assert len(set(train_row_indices) & set(test_row_indices)) == 0
assert len(set(validation_row_indices) & set(test_row_indices)) == 0
assert len(train_case_ids & validation_case_ids) == 0
assert len(train_case_ids & test_case_ids) == 0
assert len(validation_case_ids & test_case_ids) == 0

print("All final assertions passed.")
print(f"train={len(train_canonical):,}, validation={len(validation_canonical):,}, test={len(test_canonical):,}")
print("Train/Test case overlap: 0")
print("Train/Validation case overlap: 0")
print("Validation/Test case overlap: 0")

## 15. Save the clean dataset and metadata

The saved dataset contains only the canonical columns needed by the next notebook. `save_to_disk()` preserves the Hugging Face `Dataset` format and can be loaded later without rebuilding the source selection. A JSON metadata file records the source, seed, sizes, distributions, and decisions needed to audit the experiment.

Nothing is uploaded to the Hugging Face Hub.

In [ ]:
OUTPUT_DIR = Path("data") / "asclepius_5k"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_canonical.save_to_disk(str(OUTPUT_DIR / "train"))
validation_canonical.save_to_disk(str(OUTPUT_DIR / "validation"))
test_canonical.save_to_disk(str(OUTPUT_DIR / "test"))


def counts_as_json(series):
    return {str(label): int(count) for label, count in series.items()}


def dataset_task_counts(dataset):
    return pd.Series([str(value) for value in dataset["task"]]).value_counts(sort=False)

split_task_counts = {
    split_name: counts_as_json(dataset_task_counts(split_dataset))
    for split_name, split_dataset in canonical_datasets.items()
}

metadata = {
    "dataset_name": DATASET_NAME,
    "dataset_source": DATASET_SOURCE,
    "random_seed": SEED,
    "number_of_examples": TARGET_SIZE,
    "original_number_of_examples": len(raw),
    "train_size": len(train_canonical),
    "validation_size": len(validation_canonical),
    "test_size": len(test_canonical),
    "task_distribution": counts_as_json(selected_task_counts),
    "full_task_distribution": counts_as_json(full_task_counts),
    "split_task_distribution": split_task_counts,
    "selection_method": selection_method,
    "split_method": split_method,
    "case_identifier_column": case_column,
    "number_of_unique_cases_in_source": number_of_cases,
    "tokenizer_name": TOKENIZER_NAME,
    "tokenizer_context_length": TOKENIZER_CONTEXT_LENGTH,
    "tokenizer_analysis_completed": tokenizer_analysis_completed,
    "date_created": datetime.now(timezone.utc).isoformat(),
}

metadata_path = OUTPUT_DIR / "metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved artifacts:")
for path in [OUTPUT_DIR / "train", OUTPUT_DIR / "validation", OUTPUT_DIR / "test", metadata_path]:
    print(" -", path)

## Dataset Decision

The report below is generated from the variables computed above so that it reflects the actual dataset loaded and the actual selection/split results, rather than a hardcoded assumption about the source.

In [ ]:
dominant_task = str(full_task_counts.idxmax())
dominant_task_percentage = float(full_task_percentages.loc[full_task_counts.idxmax()])

print("Source dataset: Asclepius Synthetic Clinical Notes")
print(f"Original size: {len(raw):,} examples (the source is approximately 158k examples)")
print(f"Selected size: {len(selected_raw):,}")
print(f"Training: {len(train_canonical):,}")
print(f"Validation: {len(validation_canonical):,}")
print(f"Test: {len(test_canonical):,}")
print(f"Sampling: {selection_method}")
print(f"Splitting: {split_method}")
print("GPU: not required")
print("Training: NOT performed in this notebook")

## What We Learned

The next cell summarizes the findings from the executed inspection. In particular, it records the observed task count, the dominant task, whether case identifiers were repeated, the largest selection-distribution deviation, and whether tokenizer analysis was deferred.

In [ ]:
selected_deviation = float((selected_task_percentages - full_task_percentages).abs().max())
case_finding = (
    f"The source has {number_of_cases:,} unique cases; {duplicate_case_count:,} cases contain more than one row."
)
if duplicate_case_count == 0:
    case_finding += " Because every case has one row, the stratified row split is case-safe."
else:
    case_finding += " Whole-case selection and splitting were used so rows from one case stay together."

tokenizer_finding = (
    "Tokenizer analysis was deferred until the exact base model is selected."
    if not tokenizer_analysis_completed
    else f"Tokenizer analysis used {TOKENIZER_NAME!r}; context-window exceedance was computed only when the tokenizer reported a finite context length."
)

learned_summary = f"""
- The loaded source contains **{len(raw):,} rows** and **{len(full_task_counts)} task labels**. The largest observed task is **{dominant_task}** at **{dominant_task_percentage:.2f}%** of the source.
- {case_finding}
- The selected subset contains **{len(selected_raw):,} rows**. Its largest absolute task-percentage difference from the source is **{selected_deviation:.3f} percentage points**.
- The final split contains exactly **{len(train_canonical):,} train**, **{len(validation_canonical):,} validation**, and **{len(test_canonical):,} test** examples, with zero case overlap in every pairwise leakage check.
- Character/word length distributions were measured before any truncation decision; long examples were not discarded.
- {tokenizer_finding}
- The saved representation is model-independent: `task`, `clinical_note`, `instruction`, and `target`.
"""
display(Markdown(learned_summary))

## Next Notebook

### `02_qlora_medical_finetuning.ipynb`

The next notebook will:

- Select and verify the final approximately 3B medical base model.
- Load its tokenizer and apply the model's actual chat template.
- Analyze input-plus-target token lengths, including special tokens, without guessing a context window.
- Load the processed 5k dataset from `data/asclepius_5k/`.
- Configure 4-bit quantization and LoRA/QLoRA.
- Configure checkpointing and resume behavior.
- Perform a short smoke-test training run.
- Perform the full 2–3 hour Kaggle experiment.
- Evaluate the base model versus the fine-tuned adapter.
- Save the LoRA adapter.

This notebook intentionally stops before all model training steps.